# Deep Learning Chatbot — CS Copilot Intent Classification

Notebook ini menggabungkan 3 tahap utama:
1. **LANGKAH 1** — `train_dl_model.py`: Pelatihan model LSTM
2. **LANGKAH 2** — `copilot_deep.py`: Inference / penerapan model
3. **LANGKAH 3** — `test_copilot_deep.py`: Pengujian manual

---
## LANGKAH 1 — Pelatihan Model Deep Learning (`train_dl_model.py`)

In [1]:
from __future__ import annotations

import json
import re
import os
import sys
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np

try:
    import tensorflow as tf
    from tensorflow.keras.layers import Dense, Embedding, LSTM, Dropout, Bidirectional
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.preprocessing.text import Tokenizer, tokenizer_from_json
    from tensorflow.keras.callbacks import EarlyStopping
    print("TensorFlow versi:", tf.__version__)
except ImportError:
    print("[ERROR] TensorFlow belum terinstal. Jalankan: pip install tensorflow")
    sys.exit(1)

TensorFlow versi: 2.21.0


In [2]:
# Konfigurasi path output
BASE_DIR = Path(".").resolve()
OUTPUT_DIR = (BASE_DIR.parent / "chatbot_models" / "deep_learning").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH     = OUTPUT_DIR / "intent_model.keras"
TOKENIZER_PATH = OUTPUT_DIR / "intent_tokenizer.json"
LABEL_MAP_PATH = OUTPUT_DIR / "intent_label_map.json"
print("Output dir:", OUTPUT_DIR)

Output dir: C:\Users\Fahrezi\Documents\KULIAH\churn-dashboard\backend\app\chatbot_models\deep_learning


In [3]:
# Hyperparameter
VOCAB_SIZE    = 1000   # dinaikkan karena vocab CSV lebih kaya
EMBEDDING_DIM = 64     # dinaikkan untuk representasi lebih kaya
MAX_LEN       = 20
EPOCHS        = 150
BATCH_SIZE    = 32     # dinaikkan agar training lebih stabil
PATIENCE      = 15     # lebih sabar sebelum early stop


In [4]:
# TAHAP 1: Load data latih dari CSV yang sudah digenerate
import pandas as pd

DATASET_PATH = Path(".").resolve() / "intent_dataset.csv"
df_train     = pd.read_csv(DATASET_PATH)

TRAINING_SAMPLES = list(zip(df_train["text"].tolist(), df_train["intent"].tolist()))

print(f"Dataset     : {DATASET_PATH}")
print(f"Total sampel: {len(TRAINING_SAMPLES)}")
intents = sorted(df_train["intent"].unique().tolist())
print(f"Intent unik : {intents}")
print()
print("Per-intent count:")
print(df_train["intent"].value_counts().sort_index().to_string())


Dataset     : C:\Users\Fahrezi\Documents\KULIAH\churn-dashboard\backend\app\nlp_raw_model\intent_dataset.csv
Total sampel: 55000
Intent unik : ['ANALISIS_PELANGGAN', 'DRAF_EMAIL', 'FAKTOR_CHURN', 'GREETING', 'JUMLAH_RISIKO_TINGGI', 'METRIK_OVERVIEW', 'MODEL_INFO', 'SEGMEN_ANALISIS', 'STRATEGI_RETENSI', 'TREN_CHURN', 'VIP_RISK']

Per-intent count:
intent
ANALISIS_PELANGGAN      5000
DRAF_EMAIL              5000
FAKTOR_CHURN            5000
GREETING                5000
JUMLAH_RISIKO_TINGGI    5000
METRIK_OVERVIEW         5000
MODEL_INFO              5000
SEGMEN_ANALISIS         5000
STRATEGI_RETENSI        5000
TREN_CHURN              5000
VIP_RISK                5000


In [5]:
# TAHAP 2: Tokenisasi & Preprocessing
texts     = [t for t, _ in TRAINING_SAMPLES]
raw_labels = [l for _, l in TRAINING_SAMPLES]

unique_labels = sorted(list(set(raw_labels)))
label_to_idx: Dict[str, int] = {lbl: i for i, lbl in enumerate(unique_labels)}
idx_to_label: Dict[int, str] = {i: lbl for lbl, i in label_to_idx.items()}
num_classes = len(unique_labels)

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded    = pad_sequences(sequences, maxlen=MAX_LEN, padding="post", truncating="post")

labels_idx    = np.array([label_to_idx[l] for l in raw_labels])
labels_onehot = tf.keras.utils.to_categorical(labels_idx, num_classes=num_classes)

print(f"Kelas intent: {unique_labels}")
print(f"Jumlah kosakata: {len(tokenizer.word_index)} kata")
print(f"Shape data: {padded.shape}")

Kelas intent: ['ANALISIS_PELANGGAN', 'DRAF_EMAIL', 'FAKTOR_CHURN', 'GREETING', 'JUMLAH_RISIKO_TINGGI', 'METRIK_OVERVIEW', 'MODEL_INFO', 'SEGMEN_ANALISIS', 'STRATEGI_RETENSI', 'TREN_CHURN', 'VIP_RISK']
Jumlah kosakata: 493 kata
Shape data: (55000, 20)


In [6]:
# TAHAP 3: Bangun Arsitektur Model (Embedding + BiLSTM)
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN, name="word_embedding"),
    Bidirectional(LSTM(64, return_sequences=False), name="bilstm"),
    Dropout(0.3, name="dropout"),
    Dense(32, activation="relu", name="hidden_dense"),
    Dense(num_classes, activation="softmax", name="intent_output"),
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

c:\Users\Fahrezi\Documents\KULIAH\churn-dashboard\.venv\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ word_embedding (Embedding)      │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm (Bidirectional)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_dense (Dense)            │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ intent_output (Dense)           │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
# TAHAP 4: Latih Model dengan Validation Split (deteksi overfitting)
early_stop = EarlyStopping(
    monitor="val_loss",        # pantau val_loss, BUKAN training loss
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    padded, labels_onehot,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.15,     # 15% untuk validasi (330 sampel)
    callbacks=[early_stop],
    verbose=1,
)

# Evaluasi final pada seluruh data
final_loss, final_acc = model.evaluate(padded, labels_onehot, verbose=0)
print(f"\nTrain Loss    : {final_loss:.4f}")
print(f"Train Accuracy: {final_acc * 100:.2f}%")

# Diagnostik overfitting: bandingkan train vs val accuracy
train_acc_last = history.history["accuracy"][-1]
val_acc_last   = history.history["val_accuracy"][-1]
best_val_acc   = max(history.history["val_accuracy"])
print(f"\nTraining acc (epoch terakhir) : {train_acc_last:.4f}")
print(f"Validation acc (epoch terakhir): {val_acc_last:.4f}")
print(f"Best validation acc            : {best_val_acc:.4f}")
gap = train_acc_last - val_acc_last
print(f"Gap (train - val)              : {gap:.4f}")
if gap > 0.05:
    print("[PERINGATAN] Gap > 5% -> kemungkinan OVERFIT")
else:
    print("[OK] Gap kecil -> generalisasi baik")


Epoch 1/150
1461/1461 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.9431 - loss: 0.1807 - val_accuracy: 0.9999 - val_loss: 7.0076e-04
Epoch 2/150
1461/1461 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.9998 - loss: 0.0013 - val_accuracy: 0.9999 - val_loss: 7.7226e-04
Epoch 3/150
1461/1461 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.9994 - loss: 0.0025 - val_accuracy: 0.9985 - val_loss: 0.0036
Epoch 4/150
1461/1461 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.9999 - loss: 4.5708e-04 - val_accuracy: 1.0000 - val_loss: 5.0537e-05
Epoch 5/150
1461/1461 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 1.0000 - loss: 4.7027e-05 - val_accuracy: 1.0000 - val_loss: 7.4478e-05
Epoch 6/150
1461/1461 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 1.0000 - loss: 1.3540e-05 - val_accuracy: 1.0000 - val_loss: 1.6125e-06
Epoch 7/150
1461/1461 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.9999 - loss: 8.1918e-04 - val_accuracy: 1.0000 - val_loss: 3.6306e-05
Epoch 8/150
1461/1461 ━━━━━━━━━━━━━━━━

In [8]:
# TAHAP 5: Simpan Model, Tokenizer, dan Label Map
model.save(MODEL_PATH)
print(f"[SIMPAN] Model -> {MODEL_PATH}")

with open(TOKENIZER_PATH, "w", encoding="utf-8") as f:
    f.write(tokenizer.to_json())
print(f"[SIMPAN] Tokenizer -> {TOKENIZER_PATH}")

with open(LABEL_MAP_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "label_to_idx": label_to_idx,
        "idx_to_label": {str(k): v for k, v in idx_to_label.items()},
    }, f, ensure_ascii=False, indent=2)
print(f"[SIMPAN] Label map -> {LABEL_MAP_PATH}")

[SIMPAN] Model -> C:\Users\Fahrezi\Documents\KULIAH\churn-dashboard\backend\app\chatbot_models\deep_learning\intent_model.keras
[SIMPAN] Tokenizer -> C:\Users\Fahrezi\Documents\KULIAH\churn-dashboard\backend\app\chatbot_models\deep_learning\intent_tokenizer.json
[SIMPAN] Label map -> C:\Users\Fahrezi\Documents\KULIAH\churn-dashboard\backend\app\chatbot_models\deep_learning\intent_label_map.json


---
## LANGKAH 2 — Inference / Penerapan Model (`copilot_deep.py`)

In [9]:
# TAHAP 6: Muat Model dari disk
import re
from tensorflow.keras.models import load_model
from sklearn.feature_extraction.text import TfidfVectorizer

INTENT_THRESHOLD = 0.40
_DL_READY = False
_tokenizer_infer = None
_intent_model = None
_idx_to_label_infer: Dict[str, str] = {}

missing = [p for p in [MODEL_PATH, TOKENIZER_PATH, LABEL_MAP_PATH] if not p.exists()]
if missing:
    print("[PERINGATAN] File belum ada, jalankan sel pelatihan terlebih dahulu:", missing)
else:
    _intent_model = load_model(MODEL_PATH)
    with open(TOKENIZER_PATH, "r", encoding="utf-8") as f:
        _tokenizer_infer = tokenizer_from_json(f.read())
    with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
        label_data = json.load(f)
    _idx_to_label_infer = label_data["idx_to_label"]
    _DL_READY = True
    print("[INFO] Model Deep Learning berhasil dimuat.")

[INFO] Model Deep Learning berhasil dimuat.


In [10]:
# Data FAQ
FAQ_ITEMS: List[Tuple[str, str]] = [
    ("Cara bikin email penawaran diskon?",
     "Ketik: 'Tolong buatkan draf email penawaran diskon untuk pelanggan C-0001'."),
    ("Cara melihat faktor utama churn?",
     "Ketik: 'Apa faktor utama churn bulan ini?'."),
    ("Cara melihat ringkasan pelanggan?",
     "Ketik: 'Ringkasan pelanggan C-0001'."),
    ("Apa saja fitur copilot?",
     "Copilot bisa membuat draf email, merangkum faktor churn, melihat tren, analisis segmen, dan memberi ringkasan pelanggan."),
]
FAQ_THRESHOLD   = 0.32
_faq_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
_FAQ_MATRIX     = _faq_vectorizer.fit_transform([q for q, _ in FAQ_ITEMS])
print("FAQ siap.")

FAQ siap.


In [11]:
# Helper Functions
def _pick_variant(items: List[str], seed: int) -> str:
    rng = np.random.default_rng(seed)
    return items[int(rng.integers(0, len(items)))]

def _tone_for_risk(risk_label: str) -> str:
    label = (risk_label or "").lower()
    return "formal" if label == "high" else "neutral" if label == "medium" else "casual"

def _context_key(risk_label: str) -> str:
    label = (risk_label or "").lower()
    return label if label in ("high", "medium", "low") else "medium"

def extract_customer_id(message: str) -> str:
    # Ubah menjadi ini:
    match = re.search(r'c-\d+', message, re.IGNORECASE)
    return match.group(0) if match else ""

def _keyword_intent(message: str) -> str:
    text = message.lower()
    if any(k in text for k in ["halo", "hai", "hello", "hi", "hey", "selamat pagi",
                                "selamat siang", "selamat sore", "selamat malam",
                                "bisa bantu", "apa kabar", "permisi", "assalamualaikum"]):
        return "GREETING"
    if any(k in text for k in ["email", "draf", "draft", "penawaran", "retention", "kirim pesan"]):
        return "DRAF_EMAIL"
    if any(k in text for k in ["vip", "premium", "enterprise", "bernilai", "whale", "kerugian terbesar"]):
        return "VIP_RISK"
    if any(k in text for k in ["berapa", "jumlah", "total", "hitung", "count"]):
        return "JUMLAH_RISIKO_TINGGI"
    if any(k in text for k in ["strategi", "saran", "rekomendasi", "tips", "cara mencegah", "solusi"]):
        return "STRATEGI_RETENSI"
    if any(k in text for k in ["tren", "trend", "grafik", "historis", "naik turun", "pola churn"]):
        return "TREN_CHURN"
    if any(k in text for k in ["segmen", "segment", "plan type", "per paket", "monthly vs annual"]):
        return "SEGMEN_ANALISIS"
    if any(k in text for k in ["model", "algoritma", "machine learning", "akurasi", "neural"]):
        return "MODEL_INFO"
    if any(k in text for k in ["ringkasan", "overview", "summary", "kondisi", "dashboard", "laporan"]):
        return "METRIK_OVERVIEW"
    if any(k in text for k in ["analisis", "profil", "cek", "detail", "lihat data", "status customer", "c-"]):
        return "ANALISIS_PELANGGAN"
    if any(k in text for k in ["faktor", "penyebab", "feature importance", "churn", "kenapa"]):
        return "FAKTOR_CHURN"
    return "UNKNOWN"

def retrieve_faq(message: str) -> Dict[str, str]:
    text = message.strip()
    if not text:
        return {"answer": "", "score": 0.0}
    query  = _faq_vectorizer.transform([text])
    scores = (_FAQ_MATRIX @ query.T).toarray().ravel()
    best_idx   = int(np.argmax(scores))
    best_score = float(scores[best_idx])
    if best_score < FAQ_THRESHOLD:
        return {"answer": "", "score": best_score}
    return {"answer": FAQ_ITEMS[best_idx][1], "score": best_score}

def summarize_top_factors(feature_importance: Dict[str, float], top_n: int = 5) -> str:
    items = list(feature_importance.items())[:top_n]
    if not items:
        return "Belum ada data faktor utama churn."
    lines = ["Faktor utama churn bulan ini berdasarkan feature importance:"]
    for i, (name, score) in enumerate(items, start=1):
        lines.append(f"{i}. {name} (importance {score:.3f})")
    return "\n".join(lines)

print("Helper functions siap.")

Helper functions siap.


In [12]:
# TAHAP 7 & 8: Fungsi Deteksi Niat (detect_intent)
def detect_intent(message: str) -> Dict[str, str]:
    """Mendeteksi niat menggunakan BiLSTM DL atau keyword fallback."""
    text = message.strip()
    if not text:
        return {"intent": "unknown", "score": 0.0, "source": "none", "customer_id": ""}

    extracted_id    = extract_customer_id(text)
    fallback_intent = _keyword_intent(text)

    if not _DL_READY:
        return {"intent": fallback_intent, "score": 0.0, "source": "keyword_fallback", "customer_id": extracted_id}

    # TAHAP 7: Preprocessing input
    sequence = _tokenizer_infer.texts_to_sequences([text])
    padded_input = pad_sequences(sequence, maxlen=MAX_LEN, padding="post", truncating="post")

    # TAHAP 8: Prediksi intent
    probs      = _intent_model.predict(padded_input, verbose=0)[0]
    best_idx   = int(np.argmax(probs))
    best_score = float(probs[best_idx])
    best_label = _idx_to_label_infer.get(str(best_idx), "unknown")

    if best_score < INTENT_THRESHOLD:
        best_label = fallback_intent

    return {
        "intent": best_label,
        "score": best_score,
        "source": "deep_learning" if best_score >= INTENT_THRESHOLD else "keyword_fallback",
        "customer_id": extracted_id,
    }

print("detect_intent() siap.")

detect_intent() siap.


---
## LANGKAH 3 — Pengujian Manual (`test_copilot_deep.py`)

In [13]:
tests = [
    # Cek DRAF_EMAIL
    "Bikin email retention buat C-0001",
    "Tolong susun email diskon untuk C-0008",
    
    # Cek FAKTOR_CHURN
    "Faktor churn dominan minggu ini apa?",
    "Tunjukkan faktor penting churn",
    
    # Cek ANALISIS_PELANGGAN
    "Ringkasan C-0010",
    "Profil customer C-0021",
    
    # Cek GREETING
    "Bisa bantu apa?",
    "Aku bisa tanya apa saja?",
    
    # Cek MODEL_INFO
    "Model AI apa yang dipakai chatbot ini?",
    
    # Cek METRIK_OVERVIEW
    "Tolong berikan ringkasan dashboard bulan ini",
    
    # Cek STRATEGI_RETENSI
    "Saran untuk mengurangi churn rate dong"
]

print("=== TEST COPILOT DEEP LEARNING ===\n")
for msg in tests:
    result = detect_intent(msg)
    print(f"Input   : {msg}")
    print(f"Intent  : {result['intent']}  (score: {result['score']:.2%}, source: {result['source']})")
    print(f"Cust.ID : {result['customer_id'] or '-'}")
    print()

=== TEST COPILOT DEEP LEARNING ===

Input   : Bikin email retention buat C-0001
Intent  : DRAF_EMAIL  (score: 99.64%, source: deep_learning)
Cust.ID : C-0001

Input   : Tolong susun email diskon untuk C-0008
Intent  : DRAF_EMAIL  (score: 100.00%, source: deep_learning)
Cust.ID : C-0008

Input   : Faktor churn dominan minggu ini apa?
Intent  : FAKTOR_CHURN  (score: 91.12%, source: deep_learning)
Cust.ID : -

Input   : Tunjukkan faktor penting churn
Intent  : FAKTOR_CHURN  (score: 100.00%, source: deep_learning)
Cust.ID : -

Input   : Ringkasan C-0010
Intent  : ANALISIS_PELANGGAN  (score: 74.51%, source: deep_learning)
Cust.ID : C-0010

Input   : Profil customer C-0021
Intent  : ANALISIS_PELANGGAN  (score: 100.00%, source: deep_learning)
Cust.ID : C-0021

Input   : Bisa bantu apa?
Intent  : GREETING  (score: 100.00%, source: deep_learning)
Cust.ID : -

Input   : Aku bisa tanya apa saja?
Intent  : GREETING  (score: 100.00%, source: deep_learning)
Cust.ID : -

Input   : Model AI apa yang d